# DeepStack

Inteligência Artificial de nível especialista em Heads-Up No-Limit Poker

Raciocinar enquanto joga usando a “intuição” aprimorada por meio de aprendizado profundo para reavaliar sua estratégia a cada decisão .

Tentativa de replicar o que foi feito, adicionando técnicas de outras libs como Libratus e também melhorias possíveis

In [3]:
import pandas as pd

deepdata = pd.read_csv("./data/deepstack_training_data.csv")
deepdata.head()

,player,private_cards,community_cards,betting_rounds,total_pot,action_taken,outcome
0,player1,"['6 of hearts', 'A of spades']","['5 of spades', 'J of spades', 'Q of clubs', '...","[{'fold': 0.7018801043927544, 'call': 0.596648...",7.933804,fold,0.074681
1,player2,"['2 of clubs', '3 of spades']","['5 of spades', 'J of spades', 'Q of clubs', '...","[{'fold': 0.7018801043927544, 'call': 0.596648...",7.933804,fold,0.199752
2,player1,"['8 of spades', '7 of spades']","['5 of clubs', '3 of diamonds', 'Q of clubs', ...","[{'fold': 0.8324133415973884, 'call': 0.680217...",4.510486,call,0.546591
3,player2,"['K of hearts', 'Q of spades']","['5 of clubs', '3 of diamonds', 'Q of clubs', ...","[{'fold': 0.8324133415973884, 'call': 0.680217...",4.510486,raise,-0.311049
4,player1,"['9 of spades', 'J of spades']","['8 of diamonds', '3 of spades', '4 of clubs',...","[{'fold': 0.3413528458146922, 'call': 0.279417...",6.122164,raise,-0.657418


In [65]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Função de pré processamento

In [8]:
def preprocess_data(df):
    # Define a codificação para os ranks e naipes
    card_encoding = {
        "2": 2, "3": 3, "4": 4, "5": 5, "6": 6, "7": 7, "8": 8, "9": 9,
        "10": 10, "J": 11, "Q": 12, "K": 13, "A": 14,
        "hearts": 1, "diamonds": 2, "clubs": 3, "spades": 4
    }
    
    # Encode a single card
    def encode_card(card):
        if isinstance(card, list) and all(isinstance(x, int) for x in card):
            return card  # Already encoded
        if not isinstance(card, str):
            raise ValueError(f"Invalid card value: {card}")
        rank, suit = card.split(" of ")
        return [card_encoding[rank], card_encoding[suit]]
    
    # Process private cards
    def process_private_cards(cards):
        if isinstance(cards, list) and all(isinstance(x, int) for x in cards):
            return cards  # Already encoded
        if isinstance(cards, str):
            cards = eval(cards)  # Convert string to list
        if isinstance(cards, list) and len(cards) == 2:
            return encode_card(cards[0]) + encode_card(cards[1])
        else:
            raise ValueError(f"Invalid private cards: {cards}")
    
    df['private_cards'] = df['private_cards'].apply(process_private_cards)
    
    # Process community cards
    def process_community_cards(cards):
        if isinstance(cards, list) and all(isinstance(x, int) for x in cards):
            return cards  # Already encoded
        if isinstance(cards, str):
            cards = eval(cards)  # Convert string to list
        if isinstance(cards, list):
            return sum([encode_card(c) for c in cards], [])  # Flatten encoded cards
        else:
            raise ValueError(f"Invalid community cards: {cards}")
    
    df['community_cards'] = df['community_cards'].apply(process_community_cards)
    
    # Process betting rounds
    def process_betting_rounds(rounds):
        if isinstance(rounds, list) and all(isinstance(r, (float, int)) for r in rounds):
            return rounds  # Already processed
        if isinstance(rounds, str):
            rounds = eval(rounds)  # Convert string to list of dictionaries
        if isinstance(rounds, list) and all(isinstance(r, dict) for r in rounds):
            return [sum(r.values()) for r in rounds]
        else:
            raise ValueError(f"Invalid betting rounds: {rounds}")
    
    df['betting_rounds'] = df['betting_rounds'].apply(process_betting_rounds)
    
    # Combine features into a single list
    df['features'] = df.apply(
        lambda row: row['private_cards'] + row['community_cards'] + row['betting_rounds'] + [row['total_pot']],
        axis=1
    )
    
    # One-hot encode actions
    df = pd.get_dummies(df, columns=['action_taken'])
    
    # Expand features into individual columns
    features = pd.DataFrame(df['features'].to_list(), index=df.index)
    df = pd.concat([features, df.drop(columns=['private_cards', 'community_cards', 'betting_rounds', 'features'])], axis=1)
    
    return df

In [9]:
processed_data = preprocess_data(deepdata)

In [10]:
# split into features and target
X = processed_data.drop(columns=['outcome', 'player'])
y = processed_data['outcome']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Convert boolean columns to float
X_train = X_train.astype({col: 'float32' for col in X_train.select_dtypes('bool').columns})
X_test = X_test.astype({col: 'float32' for col in X_test.select_dtypes('bool').columns})
print(X_train.dtypes)

0                       int64
1                       int64
2                       int64
3                       int64
4                       int64
5                       int64
6                       int64
7                       int64
8                       int64
9                       int64
10                      int64
11                      int64
12                      int64
13                      int64
14                    float64
15                    float64
16                    float64
17                    float64
18                    float64
total_pot             float64
action_taken_call     float32
action_taken_fold     float32
action_taken_raise    float32
dtype: object


## Rede neural

Rede de "intuição"

Linear (Camada Densa)
- Redes densas são ideais para processar dados tabulares ou representações compactas do estado do jogo
- Capturam interações não-lineares entre as entradas, como relação entre cartas e apostas
- Estrutura:
    - Camada de entrada: Aceita um vetor de estado com todas as informações codificadas (ex. cartas, apostas, pote)
    - Camadas ocultas: 2-3 camadas densas com neurônios suficientes para modelar a complexidade do jogo
    - Camada de saída: Um único neurônio para prever o outcome (valor esperado do estado)

ReLU
- A ReLU (Rectified Linear Unit) acelera o aprendizado e reduz problemas de gradiente desaparecendo.
- Adequada para redes densas, pois lida bem com entradas não-normalizadas.

Perceptron Multicamadas (MLP)
- O formato 128 -> 64 -> 1 foi escolhido com base em:
    - 128 neurônios na primeira camada: Suficientes para capturar a alta dimensionalidade das entradas.
    - 64 neurônios na segunda camada: Reduz a dimensionalidade progressivamente.
    - 1 neurônio na camada de saída: Para prever o outcomeoutcome (valor esperado).

## Significado da coluna de outcome

Recompensa Simulada
- Representa o ganho ou perda do jogador em uma rodada, dependendo das cartas, apostas e ações tomadas.
- Geralmente, é um valor numérico que pode ser:
    - Positivo: Se o jogador ganhou fichas nessa rodada.
    - Negativo: Se o jogador perdeu fichas nessa rodada.
    - Zero: Se o jogador empatou ou tomou uma ação sem impacto financeiro direto.

In [ ]:
class IntuitionNetwork(nn.Module):
    def __init__(self, input_dim):
        super(IntuitionNetwork, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.fc(x)

In [9]:
# initialize model, loss and optimizer
input_dim = X_train.shape[1]
model = IntuitionNetwork(input_dim=input_dim)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [10]:
# convert data to tensors
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

In [11]:
print("X_train_tensor shape:", X_train_tensor.shape)
print("y_train_tensor shape:", y_train_tensor.shape)
print("X_test_tensor shape:", X_test_tensor.shape)
print("y_test_tensor shape:", y_test_tensor.shape)

X_train_tensor shape: torch.Size([8000, 23])
y_train_tensor shape: torch.Size([8000, 1])
X_test_tensor shape: torch.Size([2000, 23])
y_test_tensor shape: torch.Size([2000, 1])


In [25]:
# train the model
epochs = 2000
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    predictions = model(X_train_tensor)
    loss = criterion(predictions, y_train_tensor)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

Epoch 1/2000, Loss: 0.2755
Epoch 2/2000, Loss: 0.2745
Epoch 3/2000, Loss: 0.2767
Epoch 4/2000, Loss: 0.2755
Epoch 5/2000, Loss: 0.2753
Epoch 6/2000, Loss: 0.2730
Epoch 7/2000, Loss: 0.2764
Epoch 8/2000, Loss: 0.2732
Epoch 9/2000, Loss: 0.2769
Epoch 10/2000, Loss: 0.2745
Epoch 11/2000, Loss: 0.2721
Epoch 12/2000, Loss: 0.2712
Epoch 13/2000, Loss: 0.2724
Epoch 14/2000, Loss: 0.2714
Epoch 15/2000, Loss: 0.2763
Epoch 16/2000, Loss: 0.2755
Epoch 17/2000, Loss: 0.2762
Epoch 18/2000, Loss: 0.2747
Epoch 19/2000, Loss: 0.2748
Epoch 20/2000, Loss: 0.2727
Epoch 21/2000, Loss: 0.2725
Epoch 22/2000, Loss: 0.2714
Epoch 23/2000, Loss: 0.2703
Epoch 24/2000, Loss: 0.2738
Epoch 25/2000, Loss: 0.2731
Epoch 26/2000, Loss: 0.2724
Epoch 27/2000, Loss: 0.2752
Epoch 28/2000, Loss: 0.2745
Epoch 29/2000, Loss: 0.2713
Epoch 30/2000, Loss: 0.2721
Epoch 31/2000, Loss: 0.2729
Epoch 32/2000, Loss: 0.2747
Epoch 33/2000, Loss: 0.2721
Epoch 34/2000, Loss: 0.2732
Epoch 35/2000, Loss: 0.2717
Epoch 36/2000, Loss: 0.2733
E

In [28]:
model.eval()
test_predictions = model(X_test_tensor)
test_loss = criterion(test_predictions, y_test_tensor)
print(f"Test Loss: {test_loss.item():.4f}")

Test Loss: 0.4272


In [29]:
print(y_train.min(), y_train.max(), y_train.std())

-0.9999591859864374 0.9996406235746946 0.5812544990985785


### RMSE do dataset = 0.5813

In [30]:
import math
rmse = math.sqrt(test_loss.item())
print(rmse)

0.6536154315661765


### RMSE da predição do modelo = 0.5720

- Modelo está capturando boa parte da variablilidade, mas ainda tem espaço para melhorar

## DeepStack V2

In [ ]:
class PokerValueNet(nn.Module):
    def __init__(self, input_dim):
        super(PokerValueNet, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 128),  # Camada extra
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    
    def forward(self, x):
        return self.fc(x)

In [67]:
from sklearn.preprocessing import MinMaxScaler

X_train.columns = X_train.columns.astype(str)
X_test.columns = X_test.columns.astype(str)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

# Verificar os shapes
print("X_train_tensor shape:", X_train_tensor.shape)
print("X_test_tensor shape:", X_test_tensor.shape)
print("y_train_tensor shape:", y_train_tensor.shape)
print("y_test_tensor shape:", y_test_tensor.shape)

X_train_tensor shape: torch.Size([8000, 23])
X_test_tensor shape: torch.Size([2000, 23])
y_train_tensor shape: torch.Size([8000, 1])
y_test_tensor shape: torch.Size([2000, 1])


In [66]:
model = PokerValueNet(input_dim=X_train_tensor.shape[1]).to(device)

In [52]:
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
criterion = nn.MSELoss()

In [53]:
from torch.utils.data import DataLoader, TensorDataset

# create train Dataloader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [ ]:
# train
epochs = 100
for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    epoch_loss /= len(train_loader)

    # Avaliar no conjunto de teste
    model.eval()
    with torch.no_grad():
        test_predictions = model(X_test_tensor)
        test_loss = criterion(test_predictions, y_test_tensor)
    
    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {epoch_loss:.4f}, Test Loss: {test_loss.item():.4f}")

Epoch 1/100, Train Loss: 0.3396, Test Loss: 0.3223
Epoch 2/100, Train Loss: 0.3379, Test Loss: 0.3223
Epoch 3/100, Train Loss: 0.3382, Test Loss: 0.3224
Epoch 4/100, Train Loss: 0.3374, Test Loss: 0.3225
Epoch 5/100, Train Loss: 0.3378, Test Loss: 0.3226
Epoch 6/100, Train Loss: 0.3374, Test Loss: 0.3226
Epoch 7/100, Train Loss: 0.3372, Test Loss: 0.3228
Epoch 8/100, Train Loss: 0.3373, Test Loss: 0.3228
Epoch 9/100, Train Loss: 0.3377, Test Loss: 0.3229
Epoch 10/100, Train Loss: 0.3374, Test Loss: 0.3227
Epoch 11/100, Train Loss: 0.3373, Test Loss: 0.3227
Epoch 12/100, Train Loss: 0.3374, Test Loss: 0.3226
Epoch 13/100, Train Loss: 0.3366, Test Loss: 0.3231
Epoch 14/100, Train Loss: 0.3371, Test Loss: 0.3226
Epoch 15/100, Train Loss: 0.3370, Test Loss: 0.3228
Epoch 16/100, Train Loss: 0.3372, Test Loss: 0.3228
Epoch 17/100, Train Loss: 0.3374, Test Loss: 0.3226
Epoch 18/100, Train Loss: 0.3366, Test Loss: 0.3229
Epoch 19/100, Train Loss: 0.3374, Test Loss: 0.3229
Epoch 20/100, Train L

In [55]:
model.eval()
test_predictions = model(X_test_tensor)
test_loss = criterion(test_predictions, y_test_tensor)
print(f"Test Loss: {test_loss.item():.4f}")

Test Loss: 0.3328


In [56]:
print(y_train.min(), y_train.max(), y_train.std())

-0.9999591859864374 0.9996406235746946 0.5812544990985785


In [57]:
# RMSE
rmse = torch.sqrt(criterion(test_predictions, y_test_tensor))
print(f"RMSE: {rmse.item():.4f}")

# MAE
mae = torch.mean(torch.abs(test_predictions - y_test_tensor))
print(f"MAE: {mae.item():.4f}")

# R²
ss_total = torch.sum((y_test_tensor - torch.mean(y_test_tensor)) ** 2)
ss_residual = torch.sum((y_test_tensor - test_predictions) ** 2)
r2 = 1 - (ss_residual / ss_total)
print(f"R²: {r2.item():.4f}")

RMSE: 0.5769
MAE: 0.4941
R²: -0.0314


## Tunning Hiperparâmetros

In [58]:
import optuna

def objective(trial):
    # Hiperparâmetros a serem otimizados
    num_neurons = trial.suggest_int("num_neurons", 64, 256)
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)
    lr = trial.suggest_loguniform("lr", 1e-5, 1e-3)

    # Definir o modelo com parâmetros sugeridos
    class PokerValueNet(nn.Module):
        def __init__(self, input_dim):
            super(PokerValueNet, self).__init__()
            self.fc = nn.Sequential(
                nn.Linear(input_dim, 256),
                nn.ReLU(),
                nn.Dropout(0.5),
                nn.Linear(256, 128),
                nn.ReLU(),
                nn.Linear(128, 128),  # Camada extra
                nn.ReLU(),
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Linear(64, 1)
            )
    
        def forward(self, x):
            return self.fc(x)

    model = PokerValueNet(input_dim=X_train_tensor.shape[1])
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    # Treinar o modelo (simplificado)
    for epoch in range(100):  # Apenas 10 épocas para tuning rápido
        model.train()
        optimizer.zero_grad()
        predictions = model(X_train_tensor)
        loss = criterion(predictions, y_train_tensor)
        loss.backward()
        optimizer.step()

    # Avaliação no conjunto de teste
    model.eval()
    with torch.no_grad():
        test_predictions = model(X_test_tensor)
        test_loss = criterion(test_predictions, y_test_tensor)
    
    return test_loss.item()

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

# Melhor conjunto de hiperparâmetros
print(study.best_params)

[I 2024-11-21 16:21:50,712] A new study created in memory with name: no-name-62ee014a-0ff9-4b59-9886-689f2c9a5df0
/tmp/ipykernel_41390/3680314034.py:7: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-5, 1e-3)
[I 2024-11-21 16:21:52,333] Trial 0 finished with value: 0.3228951394557953 and parameters: {'num_neurons': 230, 'dropout_rate': 0.4375708848834019, 'lr': 1.7835497332678672e-05}. Best is trial 0 with value: 0.3228951394557953.
[I 2024-11-21 16:21:53,962] Trial 1 finished with value: 0.32254764437675476 and parameters: {'num_neurons': 99, 'dropout_rate': 0.18292159387495946, 'lr': 0.00012751668135957663}. Best is trial 1 with value: 0.32254764437675476.
[I 2024-11-21 16:21:55,521] Trial 2 finished with value: 0.322652667760849 and parameters: {'num_neurons': 159, 'dropout_rate': 0.29729

{'num_neurons': 150, 'dropout_rate': 0.21635715191726512, 'lr': 0.00015190718581270749}


In [72]:
import numpy as np

class CFR:
    def __init__(self, num_actions):
        self.num_actions = num_actions
        self.regrets = np.zeros(num_actions, dtype=np.float32)  # Usar float32
        self.strategy = np.ones(num_actions, dtype=np.float32) / num_actions  # Usar float32

    def get_strategy(self):
        positive_regrets = np.maximum(self.regrets, 0)
        normalizing_sum = np.sum(positive_regrets)
        if normalizing_sum > 0:
            self.strategy = positive_regrets / normalizing_sum
        else:
            self.strategy = np.ones(self.num_actions, dtype=np.float32) / self.num_actions  # Usar float32
        return self.strategy

    def update_regrets(self, action_utilities):
        for a in range(self.num_actions):
            self.regrets[a] += action_utilities[a] - np.dot(self.strategy, action_utilities)

In [73]:
class LookaheadTree:
    def __init__(self, card_value_net, num_actions, depth=3, device="cpu"):
        self.card_value_net = card_value_net
        self.num_actions = num_actions
        self.depth = depth
        self.device = device  # Define o dispositivo
        self.cfr = CFR(num_actions)

    def evaluate_leaf(self, state):
        # Use a rede neural para avaliação na folha
        state_tensor = torch.tensor(state, dtype=torch.float32, device=self.device)  # Enviar para GPU
        with torch.no_grad():
            return self.card_value_net(state_tensor).item()

    def search(self, state, depth=0):
        if depth == self.depth:
            return self.evaluate_leaf(state)

        action_utilities = torch.zeros(self.num_actions, dtype=torch.float32, device=self.device)  # Garantir float32
        for action in range(self.num_actions):
            next_state = self.simulate_action(state, action)
            action_utilities[action] = self.search(next_state, depth + 1)

        strategy = torch.tensor(self.cfr.get_strategy(), dtype=torch.float32, device=self.device)  # Garantir float32
        self.cfr.update_regrets(action_utilities.cpu().numpy())  # Atualizar CFR com valores na CPU
        return torch.dot(strategy, action_utilities).item()  # Ambos são float32

    def simulate_action(self, state, action):
        # Simula ações, agora usando PyTorch
        new_state = state.clone()
        new_state[action % len(state)] += 1  # Placeholder para lógica de transição
        return new_state

In [74]:
# Inicializar a Lookahead Tree
num_actions = 3  # Número de ações possíveis (fold, call, raise)
lookahead_tree = LookaheadTree(card_value_net=model, num_actions=num_actions, depth=3, device=device)

# Loop de treinamento
epochs = 100
for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)  # Mover para GPU
        optimizer.zero_grad()

        # Atualizar valores de estados usando LookaheadTree
        batch_values = []
        for state in batch_X:
            state_value = lookahead_tree.search(state, depth=0)  # LookaheadTree agora usa GPU
            batch_values.append(state_value)

        # Treinar a rede neural com os valores calculados
        predictions = model(batch_X)
        target_values = torch.tensor(batch_values, dtype=torch.float32, device=device).view(-1, 1)
        loss = criterion(predictions, target_values)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
    epoch_loss /= len(train_loader)

    # Avaliação no conjunto de teste
    model.eval()
    with torch.no_grad():
        test_predictions = model(X_test_tensor)
        test_loss = criterion(test_predictions, y_test_tensor)

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {epoch_loss:.4f}, Test Loss: {test_loss.item():.4f}")

/tmp/ipykernel_41390/2019185121.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  state_tensor = torch.tensor(state, dtype=torch.float32, device=self.device)  # Enviar para GPU


Epoch 1/100, Train Loss: 0.0012, Test Loss: 0.3236
Epoch 2/100, Train Loss: 0.0012, Test Loss: 0.3236
Epoch 3/100, Train Loss: 0.0011, Test Loss: 0.3236
Epoch 4/100, Train Loss: 0.0012, Test Loss: 0.3236
Epoch 5/100, Train Loss: 0.0012, Test Loss: 0.3236
Epoch 6/100, Train Loss: 0.0012, Test Loss: 0.3236
Epoch 7/100, Train Loss: 0.0012, Test Loss: 0.3236
Epoch 8/100, Train Loss: 0.0012, Test Loss: 0.3236
Epoch 9/100, Train Loss: 0.0012, Test Loss: 0.3236
Epoch 10/100, Train Loss: 0.0012, Test Loss: 0.3236
Epoch 11/100, Train Loss: 0.0012, Test Loss: 0.3236
Epoch 12/100, Train Loss: 0.0012, Test Loss: 0.3236
Epoch 13/100, Train Loss: 0.0011, Test Loss: 0.3236
Epoch 14/100, Train Loss: 0.0012, Test Loss: 0.3236
Epoch 15/100, Train Loss: 0.0012, Test Loss: 0.3236
Epoch 16/100, Train Loss: 0.0012, Test Loss: 0.3236
Epoch 17/100, Train Loss: 0.0011, Test Loss: 0.3236
Epoch 18/100, Train Loss: 0.0012, Test Loss: 0.3236
Epoch 19/100, Train Loss: 0.0012, Test Loss: 0.3236
Epoch 20/100, Train L

KeyboardInterrupt: 